# EPICGA

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.EPICGA)

class EPICGA(LinearReferenceClock):
    def postprocess(self, x):
        """Model returns gestational age in days; convert to weeks."""
        return x / 7.0



In [3]:
model = pya.models.EPICGA()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = 'epicga'
model.metadata["data_type"] = 'methylation'
model.metadata["species"] = 'Homo sapiens'
model.metadata["year"] = 2021
model.metadata["approved_by_author"] = '⌛'
model.metadata["citation"] = "Haftorn, Kristine L., et al. \"An EPIC predictor of gestational age and its application to newborns conceived by assisted reproductive technologies.\" Clinical Epigenetics 13.1 (2021): 82."
model.metadata["doi"] = "https://doi.org/10.1186/s13148-021-01055-z"
model.metadata["research_only"] = None
model.metadata["notes"] = None

## Download clock dependencies

In [5]:
supplementary_url = "https://static-content.springer.com/esm/art%3A10.1186%2Fs13148-021-01055-z/MediaObjects/13148_2021_1055_MOESM7_ESM.csv"
supplementary_file_name = "epic_ga_coefs.csv"
os.system(f"curl -sL -o {supplementary_file_name} {supplementary_url}")

0

## Load features

In [6]:
df = pd.read_csv('epic_ga_coefs.csv')
mask = df['cpgs'].astype(str).str.lower().isin(['intercept', '(intercept)'])
intercept_value = float(df.loc[mask, 's0'].iloc[0]) if mask.any() else 0.0
coef_df = df.loc[~mask].reset_index(drop=True)
model.features = coef_df['cpgs'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(coef_df['s0'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([intercept_value]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Haftorn, Kristine L., et al. "An EPIC predictor of gestational '
             'age and its application to newborns conceived by assisted '
             'reproductive technologies." Clinical Epigenetics 13.1 (2021): '
             '82.',
 'clock_name': 'epicga',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1186/s13148-021-01055-z',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2021}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg12701018', 'cg00078456', 'cg14958032', 'cg00735586', 'cg09035049', 'cg23915699', 'cg24741609', 'cg00996847', 'cg12079303', 'cg20301308', 'cg21141647', 'cg13189264', 'cg16246545', 'cg06902698', 'cg01281797', 'cg01833485', 'cg12434132',

## Basic test

In [13]:
torch.manual_seed(42)
input = torch.randn(10, len(model.features), dtype=float)
model.eval()
model.to(float)
pred = model(input)
pred

tensor([[60.5326],
        [42.4761],
        [51.9958],
        [58.1296],
        [27.9050],
        [42.0799],
        [33.6746],
        [32.6965],
        [63.3254],
        [42.7455]], dtype=torch.float64, grad_fn=<DivBackward0>)

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: epic_ga_coefs.csv
